In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM
)

from peft import LoraConfig, get_peft_model
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead

import swanlab
import json
import os
from typing import List

d:\VsCodeProj\RL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Config:
    # 模型路径 (请根据实际情况修改)
    model_name = "D:\\model\\qwen\\qwen-0.6b"  ## 或者你的本地路径 "D:\\model\\qwen\\qwen-0.6b"
    reward_model_path = "D:\\model\\qwen\\qwen-0.6b" # 训练好的RM路径
    
    # 数据路径
    data_path = "PPO\\data\\ppo_train.json" # 存放对话数据的JSON文件
    
    # 输出路径
    output_dir = "PPO\\output_ppo_qwen"
    
    device = "cuda:0"
    # 训练参数
    learning_rate = 1.4e-5
    batch_size = 1 # PPO batch size，Qwen模型较大，建议设小
    mini_batch_size = 1 # 梯度更新时的batch size
    gradient_accumulation_steps = 4
    
    # PPO 特定参数
    ppo_epochs = 2
    init_kl_coef = 0.2
    target_kl = 6.0 # 如果计算出的新旧策略之间KL散度超过这个值，说明策略更新步子过大 \
                # 会导致训练稳定，此时算法会停止当前的epoch更新(early stopping)或者增加KL系数
    gamma = 1
    lam = 0.95
    cliprange = 0.2
    cliprange_value = 0.2  # 防止价值函数更新步子过大，导致训练不稳定
    vf_coef = 0.1
    
    # 生成参数
    gen_kwargs = {
        "min_length": -1,
        "top_k": 0.0,
        "top_p": 1.0,
        "do_sample": True,
        "pad_token_id": 151643, # Qwen的pad token
        "max_new_tokens": 64,   # 生成回复的最大长度
    }

    # LoRA配置 (用于微调策略模型，减少显存占用)
    lora_r = 8
    lora_alpha = 16
    lora_dropout = 0.05


# 2. 定义 Reward Model (复用之前的定义)
class MultiDimensionRewardModel(nn.Module):
    def __init__(self, model_path, device='cuda:0'):
        super().__init__()
        self.device = device
        
        # 加载基座模型
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            dtype=torch.float16, # 使用半精度节省显存
            output_hidden_states=True,
            device_map="auto"
        )
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        
        self.hidden_size = self.model.config.hidden_size
        
        # 定义四个维度的打分头
        self.score_heads = nn.ModuleDict({
            'consistency': nn.Linear(self.hidden_size, 1),
            'relevance': nn.Linear(self.hidden_size, 1),
            'coherence': nn.Linear(self.hidden_size, 1),
            'quality': nn.Linear(self.hidden_size, 1)
        })
        
        # 加载训练好的打分头权重 (如果有)
        # 这里假设你已经保存了整个模型，或者只保存了head权重
        # 如果只保存了head，需要手动加载 state_dict

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.hidden_states[-1]
        
        # 取最后一个 token 的 hidden state
        # 注意：Qwen的padding通常在右侧，直接取 -1
        last_token_hidden = last_hidden_state[:, -1, :]
        
        # 计算四个维度的分数
        scores = {}
        for dim, head in self.score_heads.items():
            scores[dim] = head(last_token_hidden).squeeze(-1)
            
        return scores

In [3]:
MultiDimensionRewardModel.base_model_prefix

AttributeError: type object 'MultiDimensionRewardModel' has no attribute 'base_model_prefix'

In [6]:
class PPODataset(Dataset):
    def __init__(self, data_path, tokenizer):
        self.tokenizer = tokenizer
        self.data = []
        raw_data = []
        with open(data_path, 'r', encoding='utf-8') as f:
            for line in f: # 跳过空行
                if not line.strip():
                    continue
                raw_data.append(json.loads(line))
                       
        for item in raw_data:
            conversations = item["conversations"]
            # 提取历史对话作为 context
            # 我们的目标是训练模型回答最后一个 user 的问题
            history_turns = conversations[:-1]
            messages = []
            # 构造messages格式
            for turn in history_turns:
                messages.append({"role": turn["role"], "content": turn["content"]})
            
            # 使用tokenizer的 apply_chat_template 来格式化 prompt
            
            prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            
            # 将 query 拼接到 prompt 后面
            target_response = conversations[-1]["content"]
            
            self.data.append({
                "prompt": prompt,
                "response": target_response
            })
            

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# ==========================================

In [7]:
config = Config()
    
# 1. 加载 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)
# 2. 加载策略模型 (Actor) - 
# 为了节省显存，我们使用 LoRA
base_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    config.model_name,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True
)

# 配置 LoRA    
peft_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # 针对 Qwen 的常见配置
    bias="none",
    task_type="CAUSAL_LM"
)

# 将 LoRA 应用到基础模型
# 注意：AutoModelForCausalLMWithValueHead 会自动包装模型
# 我们需要先包装，再应用 LoRA，或者使用特定的方法
# 这里我们使用 trl 推荐的方式：直接加载带 Value Head 的模型，然后应用 LoRA

# critic 函数形式
model = AutoModelForCausalLMWithValueHead.from_pretrained(
    pretrained_model_name_or_path = config.model_name,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
)


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1046.17it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1134.95it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [8]:
# 3. 加载 Reward Model
reward_model = MultiDimensionRewardModel(config.reward_model_path, device='cuda:0')
reward_model.eval() # 奖励模型不需要训练

config.data_path = "D:\\VsCodeProj\\RL\\PPO\\data\\ppo_train.json"
# 4. 准备数据
dataset = PPODataset(config.data_path, tokenizer)

def collator(data):  # data: List[Dict[str, List[str]]]
    return {key: [d[key] for d in data] for key in data[0]}

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1136.34it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [10]:
dataset[2]

{'prompt': '<|im_start|>user\n你好，你知道达伦·阿伦诺夫斯基这个人吗？<|im_end|>\n<|im_start|>assistant\n知道，他是美国纽约人。<|im_end|>\n<|im_start|>user\n是的，他还是哈佛大学毕业的呢，是个高材生！<|im_end|>\n<|im_start|>assistant\n那可真不简单呢，他是犹太人，都说犹太人最聪明了。<|im_end|>\n<|im_start|>user\n都这么说，看来是真的了。<|im_end|>\n<|im_start|>assistant\n哈哈，你知道吗，他是一个编剧。<|im_end|>\n<|im_start|>user\n是的，他还是名导演，制片人呢。<|im_end|>\n<|im_start|>assistant\n那很厉害啊，他也获得过国际大奖吧？<|im_end|>\n<|im_start|>user\n是的，第14届圣丹斯电影节剧情片导演奖，第65届威尼斯国际电影节金狮奖。<|im_end|>\n<|im_start|>assistant\n好神气啊。<|im_end|>\n<|im_start|>user\n是的，你看过他的作品吗？<|im_end|>\n<|im_start|>assistant\n我看过《诺亚方舟：创世之旅》和《梦之安魂曲》。<|im_end|>\n<|im_start|>user\n《梦之安魂曲》是由达伦·阿罗诺夫斯基执导的电影。<|im_end|>\n<|im_start|>assistant\n2000年10月27日是映的，但是看的挺揪心的。<|im_end|>\n<|im_start|>user\n你不喜欢这个故事情节是吗？<|im_end|>\n<|im_start|>assistant\n是的想比来说《诺亚方舟：创世之旅》更好一些。<|im_end|>\n<|im_start|>user\n这部电影是2014年3月28日在美国上映的。<|im_end|>\n<|im_start|>assistant\n是的，它是一部动作冒险片，能迎合大众的胃口。<|im_end|>\n<|im_start|>user\n是的，尤其是我，哈哈。还记得主演是谁吗？<|im_end|>\n<|im_start|>ass

In [ ]:
print(collator(dataset[0]))

KeyError: 0

In [9]:
base_model = AutoModelForCausalLMWithValueHead.from_pretrained(
        config.model_name,
        # device_map="auto",
        # dtype=torch.float16,
        # trust_remote_code=True
    )

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 7099.66it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [11]:
base_model

AutoModelForCausalLMWithValueHead(
  (pretrained_model): Qwen3ForCausalLM(
    (model): Qwen3Model(
      (embed_tokens): Embedding(151936, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          )
          (mlp): Qwen3MLP(
            (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
            (act_fn): SiLUActivation()
        

In [12]:
from transformers import AutoModelForSequenceClassification

cla_model = AutoModelForSequenceClassification.from_pretrained(
    config.model_name, num_labels=4
)

cla_model

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6813.89it/s]
Qwen3ForSequenceClassification LOAD REPORT from: D:\model\qwen\qwen-0.6b
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Qwen3ForSequenceClassification(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_a

In [18]:
base_model = AutoModelForCausalLMWithValueHead.from_pretrained(
        config.model_name,
        generation_config=AutoModelForCausalLM.from_pretrained(config.model_name
    ).generation_config
        # device_map="auto",
        # dtype=torch.float16,
        # trust_remote_code=True
    )

cf = AutoModelForCausalLM.from_pretrained(config.model_name
    ).generation_config
cf

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 6407.73it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 7824.16it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 8079.96it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warnin

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "temperature": 0.6,
  "top_k": 20,
  "top_p": 0.95
}

In [15]:
model.pretrained_model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [5]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
# model.pretrained_model.generation_config
from trl import AutoModelForCausalLMWithValueHead

config = Config()
base_model = AutoModelForCausalLMWithValueHead.from_pretrained(
        pretrained_model_name_or_path=config.model_name,
        offload_folder = "offload",
        quantization_config=bnb_config
        # device_map="auto",
        # dtype=torch.float16,
        # trust_remote_code=True
    )

base_model

Loading weights: 100%|██████████| 311/311 [00:01<00:00, 259.54it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


AutoModelForCausalLMWithValueHead(
  (pretrained_model): Qwen3ForCausalLM(
    (model): Qwen3Model(
      (embed_tokens): Embedding(151936, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear4bit(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
            (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          )
          (mlp): Qwen3MLP(
            (gate_proj): Linear4bit(in_features=1024, out_features=3072, bias=False)
            (up_proj): Linear4bit(in_features=1024, out_features=3072, bias=False)
            (down_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
            (act_fn

In [ ]:
MultiDimensionScoreModel(
  (model): Qwen3ForCausalLM(
    (model): Qwen3Model(
      (embed_tokens): Embedding(151936, 1024)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          )
          (mlp): Qwen3MLP(
            (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
            (act_fn): SiLUActivation()
          )
          (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
          (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        )
      )
      (norm): Qwen3RMSNorm((1024,), eps=1e-06)
      (rotary_emb): Qwen3RotaryEmbedding()
    )
    (lm_head): Linear(in_features=1024, out_features=151936, bias=False)
  )
  (score_heads): ModuleDict(
    (consistency): Linear(in_features=1024, out_features=1, bias=True)
    (relevance): Linear(in_features=1024, out_features=1, bias=True)
    (coherence): Linear(in_features=1024, out_features=1, bias=True)
    (quality): Linear(in_features=1024, out_features=1, bias=True)
  )
)

ModuleNotFoundError: No module named 'RL'

In [6]:
import torch

reward = torch.randn(16, 768, 1)
data = [767, 767, 698, 767, 767, 690, 767, 767, 767, 767, 767, 734, 767, 767, 767, 767]
tensor_data = torch.tensor(data)
print(tensor_data)
reward[
    torch.arange(reward.size(0)),
    tensor_data].squeeze(-1)

tensor([767, 767, 698, 767, 767, 690, 767, 767, 767, 767, 767, 734, 767, 767,
        767, 767])


tensor([-0.7691,  0.4022, -0.2515,  0.4371, -0.2145,  1.0619,  0.1193, -0.9163,
         0.4284,  1.0871,  0.5323, -0.1729, -0.6692,  0.3312, -2.7662, -1.6118])